<a href="https://colab.research.google.com/github/rudalshan0412-code/Intent_Classifier-RAG_Chatbot/blob/main/03)_RAG_%EB%AC%B8%EC%84%9C_%EB%A1%9C%EB%94%A9%2C_%EC%A0%84%EC%B2%98%EB%A6%AC%2C_%EC%B2%AD%ED%82%B9_%EA%B5%AC%ED%98%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Drive 연결 및 확인

from google.colab import drive

drive.mount("/content/drive")

%cd /content/drive/MyDrive/rag_intent_chatbot

!pwd
!find . -maxdepth 3 -type f | sort

Mounted at /content/drive
/content/drive/MyDrive/rag_intent_chatbot
/content/drive/MyDrive/rag_intent_chatbot
./data/documents/sample.txt
./data/intents.json
./main.py
./models/intent_classifier.pt
./requirements.txt
./src/chatbot.py
./src/__init__.py
./src/intent/dataset.py
./src/intent/__init__.py
./src/intent/model.py
./src/intent/predict.py
./src/intent/train.py
./src/__pycache__/__init__.cpython-312.pyc
./src/rag/chunker.py
./src/rag/document_loader.py
./src/rag/__init__.py
./src/rag/retriever.py
./src/rag/text_preprocessor.py


In [ ]:
%%writefile data/documents/sample.txt
RAG의 정의

RAG는 Retrieval-Augmented Generation의 약자로, 검색 증강 생성이라고 부른다. 일반적인 생성형 언어 모델은 학습 과정에서 익힌 지식과 현재 입력된 문맥을 바탕으로 답변을 만든다. 반면 RAG는 사용자의 질문과 관련된 외부 문서를 먼저 검색하고, 검색된 내용을 언어 모델의 입력 문맥에 함께 제공한 뒤 답변을 생성한다. 따라서 모델이 모든 정보를 내부 파라미터에 기억하고 있지 않더라도 프로젝트 문서, 수업 자료, 매뉴얼, 보고서와 같은 별도의 지식 저장소를 활용할 수 있다. 이 프로젝트에서는 문서를 읽고, 정리하고, 작은 단위로 나눈 뒤, 질문과 관련된 부분을 찾아 챗봇 답변에 활용하는 전체 과정을 직접 구현한다.

RAG가 필요한 이유

언어 모델은 그럴듯한 문장을 잘 만들지만, 학습하지 않은 최신 정보나 사용자의 개인 문서에 대해서는 정확히 알 수 없다. 또한 학습 데이터에 비슷한 내용이 있더라도 출처가 불분명하거나 세부 수치를 잘못 말하는 환각 현상이 발생할 수 있다. RAG를 적용하면 답변에 사용될 근거를 외부 문서에서 가져올 수 있으므로, 제한된 도메인에서는 답변의 정확성과 설명 가능성을 높일 수 있다. 예를 들어 학교 규정 챗봇이라면 학칙 문서에서 관련 조항을 검색하고, 제품 지원 챗봇이라면 사용 설명서에서 문제 해결 절차를 검색하도록 만들 수 있다. 문서가 수정되었을 때 모델 전체를 다시 학습하지 않고 검색 대상 문서만 갱신할 수 있다는 점도 중요한 장점이다.

문서 청킹의 의미

긴 문서를 그대로 하나의 데이터로 저장하면 사용자의 짧은 질문과 문서 전체를 비교해야 하므로 관련 부분을 정확히 찾기 어렵다. 또한 언어 모델에 문서 전체를 전달하면 입력 토큰이 지나치게 많아지고, 실제 답변에 필요하지 않은 내용까지 포함될 수 있다. 그래서 RAG에서는 문서를 여러 개의 작은 조각으로 나누는데, 이 조각을 Chunk라고 한다. 좋은 Chunk는 하나의 주제나 설명 흐름을 가능한 한 유지하면서도 검색에 사용할 수 있을 정도로 짧아야 한다. 문단 경계와 문장 경계를 고려하면 의미가 중간에서 끊기는 문제를 줄일 수 있다. 너무 긴 문단이나 문장은 최종적으로 글자 수 기준으로 나눌 수 있으며, 인접 Chunk 사이에 일부 내용을 겹치게 두면 경계 부근의 문맥이 사라지는 문제를 완화할 수 있다.

임베딩의 의미

컴퓨터는 문장의 의미를 사람처럼 직접 이해하지 못하므로, 텍스트를 수치 벡터로 변환하는 과정이 필요하다. 임베딩은 단어, 문장, 문서 조각의 의미적 특징을 여러 차원의 숫자로 표현한 벡터이다. 의미가 비슷한 문장들은 임베딩 공간에서 서로 가까운 위치에 놓이도록 학습된다. 예를 들어 '문서를 왜 나누나요?'와 '청킹이 필요한 이유는 무엇인가요?'는 사용하는 단어가 완전히 같지 않더라도 비슷한 의미를 가지므로 가까운 벡터가 되는 것을 기대할 수 있다. 이 프로젝트의 다음 단계에서는 각 Chunk를 임베딩 모델에 입력하여 벡터로 변환하고, Chunk의 원문 및 metadata와 함께 저장할 예정이다. 질문 역시 같은 임베딩 방식으로 벡터화해야 서로 비교할 수 있다.

벡터 유사도 검색

질문과 모든 Chunk가 벡터로 변환되면 두 벡터가 얼마나 비슷한지 계산할 수 있다. 대표적인 방법은 코사인 유사도이며, 두 벡터가 가리키는 방향이 비슷할수록 높은 값을 가진다. 사용자의 질문 벡터와 각 Chunk 벡터의 유사도를 계산한 뒤 점수가 높은 순서로 정렬하면, 질문에 가장 관련된 문서 조각을 선택할 수 있다. 이 과정을 벡터 유사도 검색이라고 한다. 실제 시스템에서는 문서가 매우 많을 수 있으므로 FAISS나 벡터 데이터베이스를 사용해 검색 속도를 높이기도 한다. 현재 프로젝트에서는 먼저 구조와 원리를 이해하기 위해 작은 문서 집합을 대상으로 검색 과정을 구현한 뒤, 필요하면 더 큰 저장 방식으로 확장한다. 검색 결과에는 원문뿐 아니라 source, 파일 경로, Chunk 번호와 같은 metadata도 함께 반환되어야 한다.

Intent Classifier의 역할

모든 사용자 입력에 RAG 검색이 필요한 것은 아니다. '안녕하세요'와 같은 인사, '고마워'와 같은 감사 표현, '너는 누구야?'와 같은 챗봇 정보 질문은 미리 정의된 일반 응답으로 처리할 수 있다. 반면 'RAG에서 임베딩은 어떤 역할을 하나요?'처럼 프로젝트 문서의 내용을 묻는 질문은 문서 검색이 필요하다. PyTorch Intent Classifier는 사용자의 문장을 greeting, goodbye, thanks, help, bot_info, document_query 중 하나로 분류한다. 최고 예측 확률이 임계값보다 낮으면 fallback으로 처리하며, document_query로 예측된 경우에만 requires_rag 값을 참으로 설정한다. 이 구조를 사용하면 단순 대화와 문서 기반 질의를 구분하여 불필요한 검색과 모델 호출을 줄일 수 있다.

현재 프로젝트의 전체 흐름

사용자가 질문을 입력하면 먼저 Intent Predictor가 문장의 의도를 예측한다. 일반 대화 인텐트라면 intents.json에 정의된 응답 중 하나를 선택해 반환한다. document_query라면 질문을 임베딩 벡터로 변환하고, 미리 저장된 Chunk 벡터와 유사도를 비교해 관련 문서 조각을 검색한다. 검색된 Chunk의 text와 metadata는 이후 답변 생성 모듈로 전달된다. 답변 생성 단계에서는 질문과 검색 근거를 함께 프롬프트에 넣어 근거 중심의 응답을 만들 수 있다. 현재 단계에서는 이 전체 구조 중 가장 앞부분인 문서 로딩, 공백 및 줄바꿈 전처리, 문단과 문장을 고려한 청킹을 구현한다. 이후에는 임베딩 생성, 벡터 저장소, Retriever, Intent Classifier와 RAG 라우팅, 최종 챗봇 인터페이스 순서로 확장할 예정이다.

Overwriting data/documents/sample.txt


In [ ]:
%%writefile src/rag/document_loader.py
"""텍스트 문서를 읽어 Document 객체로 변환한다."""

from dataclasses import dataclass
from pathlib import Path
from typing import Any


@dataclass
class Document: # @dataclass가 알아서 Document(text, metadata로 만들어준다)
    """문서 내용과 출처 정보를 함께 저장한다."""

    text: str
    metadata: dict[str, Any]


def load_text_file(file_path: str | Path) -> Document:
    """하나의 UTF-8 텍스트 파일을 불러온다."""

    path = Path(file_path) # 경로파일로 변환

    if not path.exists():
        raise FileNotFoundError(
            f"파일을 찾을 수 없습니다: {path}"
        )

    if not path.is_file():
        raise ValueError(
            f"파일 경로가 아닙니다: {path}"
        )

    if path.suffix.lower() != ".txt": # 확장자
        raise ValueError(
            "현재는 .txt 파일만 지원합니다."
        )

    text = path.read_text(encoding="utf-8")

    if not text.strip():
        raise ValueError(
            f"빈 문서입니다: {path}"
        )

    metadata = {
        "source": path.name, # 파일 이름 + 확장자
        "file_name": path.name, # 파일 이름 + 확장자
        "file_path": str(path.resolve()),  # 절대 경로(정확한 위치)
        "file_extension": path.suffix.lower(), # 확장자
    }

    return Document(
        text=text,
        metadata=metadata,
    )


def load_documents(
    directory_path: str | Path,
) -> list[Document]:
    """디렉터리 아래의 모든 .txt 파일을 불러온다."""

    directory = Path(directory_path) # 경로 파일로 변환

    if not directory.exists():
        raise FileNotFoundError(
            f"디렉터리를 찾을 수 없습니다: {directory}"
        )

    if not directory.is_dir():
        raise ValueError(
            f"디렉터리 경로가 아닙니다: {directory}"
        )

    file_paths = sorted(directory.rglob("*.txt")) # 모든 하위경로 내에 존재하는 .txt 파일의 경로 찾아서 정렬

    if not file_paths:
        raise ValueError(
            "불러올 .txt 파일이 없습니다."
        )

    documents: list[Document] = []

    for file_path in file_paths:
        try:
            document = load_text_file(file_path)
            documents.append(document)

        except ValueError as error:
            print(f"[건너뜀] {error}")

    if not documents:
        raise ValueError(
            "정상적으로 불러온 문서가 없습니다."
        )

    return documents

Overwriting src/rag/document_loader.py


In [ ]:
# 테스트
from src.rag.document_loader import (
    load_text_file,
    load_documents,
)

document = load_text_file(
    "data/documents/sample.txt"
)

print("문서 길이:", len(document.text))
print("metadata:", document.metadata)
print("내용 일부:", document.text[:100])

문서 길이: 2870
metadata: {'source': 'sample.txt', 'file_name': 'sample.txt', 'file_path': '/content/drive/MyDrive/rag_intent_chatbot/data/documents/sample.txt', 'file_extension': '.txt'}
내용 일부: RAG의 정의

RAG는 Retrieval-Augmented Generation의 약자로, 검색 증강 생성이라고 부른다. 일반적인 생성형 언어 모델은 학습 과정에서 익힌 지식과 현


In [ ]:
%%writefile src/rag/text_preprocessor.py
"""RAG 문서의 공백과 줄바꿈을 정리한다."""

import re


def normalize_whitespace(text: str) -> str:
    """공백과 줄바꿈 형식을 통일한다."""

    if not isinstance(text, str):
        raise TypeError(
            "text는 문자열이어야 합니다."
        )

    # Windows 줄바꿈을 일반 줄바꿈으로 통일
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # 탭을 일반 공백으로 변경
    text = text.replace("\t", " ")

    # 각 줄 내부의 연속 공백 정리
    lines = [
        re.sub(r" +", " ", line).strip() # re.sub(무엇을, 어떻게 바꿔주세요, 어떤 파일에서])
        for line in text.split("\n")
    ]

    text = "\n".join(lines) # lines 를 \n을 구분자로 합침

    # 빈 줄이 지나치게 많으면 두 줄로 축소
    text = re.sub(r"\n{3,}", "\n\n", text) # \n{3, } 줄바꿈이 3번 이상 반복되는 구간(일종의 정규표현식)

    return text.strip()


def preprocess_text(text: str) -> str:
    """RAG 청킹 전에 사용할 전체 전처리를 수행한다."""

    # 파일 맨 앞에 생길 수 있는 BOM 문자 제거
    text = text.replace("\ufeff", "") # 어떤 파일인지 알려주기 위해 붙이는 일종의 표식(\ufeff)

    return normalize_whitespace(text)

Overwriting src/rag/text_preprocessor.py


In [ ]:
# 테스트

from src.rag.text_preprocessor import (
    normalize_whitespace,
    preprocess_text,
)

messy_text = (
    "\ufeff  RAG는    문서를 검색합니다.\r\n"
    "\r\n"
    "\r\n"
    "\t문단 구조는 유지합니다.   "
)

print("원본:")
print(repr(messy_text))

print("\n전처리 결과:")
print(preprocess_text(messy_text))

원본:
'\ufeff  RAG는    문서를 검색합니다.\r\n\r\n\r\n\t문단 구조는 유지합니다.   '

전처리 결과:
RAG는 문서를 검색합니다.

문단 구조는 유지합니다.


In [ ]:
%%writefile src/rag/chunker.py
"""문서를 작은 Chunk 단위로 나눈다."""

from dataclasses import dataclass
from pathlib import Path
from typing import Any
import re

from .document_loader import Document


@dataclass
class Chunk:
    """검색과 임베딩에 사용할 문서 조각."""

    text: str
    chunk_id: str
    metadata: dict[str, Any]
    source: str
    start_index: int
    end_index: int


def _validate_options(
    chunk_size: int,
    chunk_overlap: int,
) -> None:
    """chunk_size와 chunk_overlap을 검사한다."""

    if chunk_size <= 1:
        raise ValueError(
            "chunk_size는 1보다 커야 합니다."
        )

    if chunk_overlap < 0:
        raise ValueError(
            "chunk_overlap은 0 이상이어야 합니다."
        )

    if chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap은 chunk_size보다 작아야 합니다."
        )


def _find_chunk_end(
    text: str,
    start_index: int,
    chunk_size: int,
) -> int:
    """문단, 문장, 글자 수 순서로 Chunk 끝을 찾는다.
       이 때, chunk는 기본적인 크기가 정해져 있으며 자연스러운 경계를 찾아 끊는 것을 목표로 한다."""

    max_end = min(
        start_index + chunk_size,
        len(text),
    )
    # 슬라이싱 시 len(text)를 넘어가지 않게 하기 위함

    # 마지막 Chunk라면 남은 텍스트 전체 사용
    if max_end == len(text):
        return max_end

    # max_end = start_index + chunk+size 인 경우
    # 지나치게 짧게 잘리지 않도록 뒤쪽 절반에서 경계 검색
    search_start = start_index + chunk_size // 2
    search_area = text[search_start:max_end]

    # 1순위: 문단 경계
    paragraph_position = search_area.rfind("\n\n") # rfind() -> 오른쪽부터 찾아라(최대한 chunk_size에 가깝게 찾기 위함)

    if paragraph_position != -1: # rfind는 찾는 글자가 없을 경우 -1을 반환한다
        return search_start + paragraph_position

    # 2순위: 문장 경계
    sentence_matches = list(
        re.finditer( # re.finditer는 해당하는 모든 위치를 찾아준다
            r"[.!?。！？](?=\s|$)",
            search_area,
        )
    )

    if sentence_matches:
        return (
            search_start
            + sentence_matches[-1].end() # 인덱스 숫자를 표기해주기 위해서는 .end()가 필요하다(복합 정보가 담긴 객체이기에)
        )

    # 3순위: 글자 수 기준
    return max_end


def chunk_text(
    text: str,
    metadata: dict[str, Any] | None = None,
    chunk_size: int = 500,
    chunk_overlap: int = 100,
) -> list[Chunk]:
    """텍스트를 overlap이 있는 Chunk 목록으로 변환한다."""

    _validate_options(
        chunk_size,
        chunk_overlap,
    )

    if not text.strip():
        return []

    metadata = dict(metadata or {}) #만약 metadata가 비었을 경우 빈 딕셔너리 생성

    source = str(
        metadata.get("source", "unknown") # 만약 "source"가 반환할 값이 없다면 "unkown" 반환
    )

    source_name = Path(source).stem or "document" # .stem은 파일이름만 반환

    chunks: list[Chunk] = []
    start_index = 0

    while start_index < len(text):
        end_index = _find_chunk_end(
            text=text,
            start_index=start_index,
            chunk_size=chunk_size,
        )

        # Chunk 앞뒤 공백 제거 및 위치 보정
        actual_start = start_index
        actual_end = end_index

        while (
            actual_start < actual_end
            and text[actual_start].isspace()
        ):
            actual_start += 1

        while (
            actual_end > actual_start
            and text[actual_end - 1].isspace()
        ):
            actual_end -= 1

        if actual_start < actual_end:
            chunk_number = len(chunks)

            chunk_id = (
                f"{source_name}_chunk_"
                f"{chunk_number:04d}"
            )

            chunk_metadata = dict(metadata)

            chunk_metadata.update(
                {
                    "chunk_id": chunk_id,
                    "chunk_index": chunk_number,
                    "start_index": actual_start,
                    "end_index": actual_end,
                }
            )

            chunks.append(
                Chunk(
                    text=text[actual_start:actual_end],
                    chunk_id=chunk_id,
                    metadata=chunk_metadata,
                    source=source,
                    start_index=actual_start,
                    end_index=actual_end,
                )
            )

        # 문서 끝까지 처리했다면 반복 종료
        if end_index >= len(text):
            break

        # 앞 Chunk의 마지막 부분을 다음 Chunk에 다시 포함
        next_start = max(
            0,
            end_index - chunk_overlap,
        )

        # 가능하면 단어 중간에서 시작하지 않도록 조정
        while (
            next_start < end_index
            and not text[next_start].isspace()
        ):
            next_start += 1

        while (
            next_start < len(text)
            and text[next_start].isspace()
        ):
            next_start += 1

        # 반복 위치가 전진하지 않는 문제 방지
        if next_start <= start_index:
            next_start = end_index

        start_index = next_start

    return chunks


def chunk_document(
    document: Document,
    chunk_size: int = 500,
    chunk_overlap: int = 100,
) -> list[Chunk]:
    """하나의 Document를 Chunk 목록으로 변환한다."""

    return chunk_text(
        text=document.text,
        metadata=document.metadata,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )


def chunk_documents(
    documents: list[Document],
    chunk_size: int = 500,
    chunk_overlap: int = 100,
) -> list[Chunk]:
    """여러 Document를 한 번에 청킹한다."""

    all_chunks: list[Chunk] = []

    for document in documents:
        document_chunks = chunk_document(
            document=document,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

        all_chunks.extend(document_chunks)

    return all_chunks

Overwriting src/rag/chunker.py


In [ ]:
# 통합 테스트

from src.rag.document_loader import (
    Document,
    load_documents,
)
from src.rag.text_preprocessor import preprocess_text
from src.rag.chunker import chunk_document


CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

documents = load_documents(
    "data/documents"
)

print("=" * 70)
print("불러온 문서 수:", len(documents))
print("=" * 70)

all_chunks = []

for document in documents:
    original_text = document.text

    processed_text = preprocess_text(
        original_text
    )

    processed_document = Document(
        text=processed_text,
        metadata=document.metadata,
    )

    chunks = chunk_document(
        document=processed_document,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )

    all_chunks.extend(chunks)

    print(
        "\nsource:",
        document.metadata["source"],
    )
    print(
        "원본 문서 길이:",
        len(original_text),
    )
    print(
        "전처리 후 길이:",
        len(processed_text),
    )
    print(
        "생성된 Chunk 수:",
        len(chunks),
    )

    for chunk in chunks:
        preview = chunk.text[:100].replace(
            "\n",
            " ",
        )

        print("\n" + "-" * 70)
        print("chunk_id:", chunk.chunk_id)
        print("글자 수:", len(chunk.text))
        print("source:", chunk.source)
        print(
            "위치:",
            chunk.start_index,
            "~",
            chunk.end_index,
        )
        print("내용 일부:", preview, "...")

불러온 문서 수: 1

source: sample.txt
원본 문서 길이: 2870
전처리 후 길이: 2869
생성된 Chunk 수: 9

----------------------------------------------------------------------
chunk_id: sample_chunk_0000
글자 수: 382
source: sample.txt
위치: 0 ~ 382
내용 일부: RAG의 정의  RAG는 Retrieval-Augmented Generation의 약자로, 검색 증강 생성이라고 부른다. 일반적인 생성형 언어 모델은 학습 과정에서 익힌 지식과 현 ...

----------------------------------------------------------------------
chunk_id: sample_chunk_0001
글자 수: 471
source: sample.txt
위치: 284 ~ 755
내용 일부: 수 있다. 이 프로젝트에서는 문서를 읽고, 정리하고, 작은 단위로 나눈 뒤, 질문과 관련된 부분을 찾아 챗봇 답변에 활용하는 전체 과정을 직접 구현한다.  RAG가 필요한 이유   ...

----------------------------------------------------------------------
chunk_id: sample_chunk_0002
글자 수: 498
source: sample.txt
위치: 657 ~ 1155
내용 일부: 문제 해결 절차를 검색하도록 만들 수 있다. 문서가 수정되었을 때 모델 전체를 다시 학습하지 않고 검색 대상 문서만 갱신할 수 있다는 점도 중요한 장점이다.  문서 청킹의 의미   ...

----------------------------------------------------------------------
chunk_id: sample_chunk_0003
글자 수: 491
source: sample.txt
위치: 1059 ~ 1550
내용 일부: 너무 긴 문단

In [ ]:
# 기본 조건 검사

from src.rag.chunker import chunk_text


assert len(all_chunks) > 1

assert all(
    chunk.text.strip()
    for chunk in all_chunks
)

assert all(
    len(chunk.text) <= CHUNK_SIZE
    for chunk in all_chunks
)

assert chunk_text(
    "",
    chunk_size=500,
    chunk_overlap=100,
) == []

print("모든 테스트를 통과했습니다.")

모든 테스트를 통과했습니다.


In [ ]:
# 청킹 설정 오류 테스트

from src.rag.chunker import chunk_text


test_cases = [
    (1, 0),
    (500, -1),
    (500, 500),
]

for chunk_size, chunk_overlap in test_cases:
    try:
        chunk_text(
            text="테스트 문서",
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

    except ValueError as error:
        print(error)

chunk_size는 1보다 커야 합니다.
chunk_overlap은 0 이상이어야 합니다.
chunk_overlap은 chunk_size보다 작아야 합니다.
